In [ ]:
# !pip install statsmodels
# Load libraries 
import pandas as pd
import numpy as np 
from math import sqrt 
import sys 
print(sys.executable)
import statsmodels.api as sm
from scipy.stats import norm
from statsmodels.tsa.stattools import adfuller

/Users/paridhiagarwal/DSE4211 Project/btc-rv-prediction/.venv/bin/python


In [16]:
# Load all predictions CV for all the models here 
LSTM_PRED_PATH = "../forecast evaluations/lstm_outputs/lstm_rolling_oos_predictions.csv"
HAR_PRED_PATH  = "../forecast evaluations/har_outputs/har_family_rolling_oos_predictions.csv"
LASSO_PRED_PATH = "../forecast evaluations/lasso_outputs/lasso_rolling_oos_predictions.csv"
RIDGE_PRED_PATH = "../forecast evaluations/ridge_outputs/ridge_rolling_oos_predictions.csv"
SVR_PRED_PATH = "../forecast evaluations/svr_outputs/svr_rolling_oos_predictions.csv"
XGB_PRED_PATH = "../forecast evaluations/xgb_outputs/xgb_rolling_oos_predictions.csv"
RF_PRED_PATH = "../forecast evaluations/rf_outputs/rf_rolling_oos_predictions.csv"

 
BENCHMARK = "HAR-RV"
HORIZONS = [1, 3, 5, 7]

In [17]:
# Load paths
def load_lstm_preds(path):
    df = pd.read_csv(path)
    # expected: date, h, y_true, y_pred (may also include error)
    df = df.copy()
    df["model"] = "LSTM"
    df["date"] = pd.to_datetime(df["date"],dayfirst=True, errors = "raise")
    df["h"] = df["h"].astype(int)
    df = df.rename(columns={"y_true": "actual", "y_pred": "predicted"})
    keep = ["date", "h", "model", "actual", "predicted"]
    return df[keep]


def load_preds(path):
    df = pd.read_csv(path)
    # expected: date, model, h, y_true, y_pred
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"],dayfirst=True, errors = "raise")
    df["h"] = df["h"].astype(int)
    df = df.rename(columns={"y_true": "actual", "y_pred": "predicted"})
    keep = ["date", "h", "model", "actual", "predicted"]
    return df[keep]

In [18]:
def dm_test_hac(loss_model, loss_bench, hac_lag):
    d = np.asarray(loss_model - loss_bench, dtype=float)
    d = d[~np.isnan(d)]
    T = len(d)

    if T < 20:
        return {"n": T, "dm_stat": np.nan, "p_value": np.nan, "mean_d": np.nan}
    
    # DM test via HAC robust intercept test on mean(d) 
    X = np.ones((T, 1))
    res = sm.OLS(d, X).fit(cov_type="HAC", cov_kwds={"maxlags": int(hac_lag)})
    dm_stat = float(res.tvalues[0])
    p_value = round(float(2.0 * (1.0 - norm.cdf(abs(dm_stat)))),6)  # two-sided
    mean_d = float(np.mean(d))

    return {
        "n": T,
        "dm_stat": dm_stat,
        "p_value": p_value,
        "mean_d": mean_d
    }


In [19]:
def adf_check_loss_diff(preds_long, benchmark=BENCHMARK, horizons=HORIZONS, alpha=0.05):
    if benchmark not in preds_long["model"].unique():
        raise ValueError(f"Benchmark '{benchmark}' not found in preds_long['model'].")

    out_rows = []
    models = sorted([m for m in preds_long["model"].unique() if m != benchmark])

    for h in horizons:
        bench = preds_long[
            (preds_long["model"] == benchmark) & (preds_long["h"] == h)
        ][["date", "actual", "predicted"]].rename(columns={"predicted": "pred_b", "actual": "actual_b"})

        if bench.empty:
            continue

        for m in models:
            other = preds_long[
                (preds_long["model"] == m) & (preds_long["h"] == h)
            ][["date", "actual", "predicted"]].rename(columns={"predicted": "pred_m"})

            merged = bench.merge(other, on=["date"], how="inner")

            actual = merged["actual_b"]
            # squared forecast error from candidate model
            loss_m = (actual - merged["pred_m"]) ** 2
            # squared forecast error from benchmark model 
            loss_b = (actual - merged["pred_b"]) ** 2

            # computing loss differentiatial (candidate model - benchmark)
            d = np.asarray(loss_m - loss_b, dtype=float)
            d = d[~np.isnan(d)]
            T = len(d)

            if T < 20:
                out_rows.append({
                    "h": h,
                    "model": m,
                    "benchmark": benchmark,
                    "mean_d": np.nan,
                    "adf_stat": np.nan,
                    "adf_p_value": np.nan,
                    "stationary_5pct": "Too few obs"
                })
                continue

            try:
                adf_res = adfuller(d, regression="c", autolag="AIC")
                adf_stat = float(adf_res[0])
                adf_p_value = float(adf_res[1])
                stationary_5pct = "Yes" if adf_p_value < alpha else "No"
            except Exception:
                adf_stat = np.nan
                adf_p_value = np.nan
                stationary_5pct = "Error"

            out_rows.append({
                "h": h,
                "model": m,
                "benchmark": benchmark,
                "n": T,
                "mean_d": float(np.mean(d)),
                "adf_stat": adf_stat,
                "adf_p_value": adf_p_value,
                "stationary_5pct": stationary_5pct
            })

    return pd.DataFrame(out_rows).sort_values(["h", "model"]).reset_index(drop=True) 

In [20]:
def dm_table_vs_benchmark(preds_long, benchmark=BENCHMARK, horizons=HORIZONS):
    if benchmark not in preds_long["model"].unique():
        raise ValueError(f"Benchmark '{benchmark}' not found in preds_long['model'].")

    out_rows = []
    models = sorted([m for m in preds_long["model"].unique() if m != benchmark])

    for h in horizons:
        bench = preds_long[(preds_long["model"] == benchmark) & (preds_long["h"] == h)][
            ["date", "actual", "predicted"]
        ].rename(columns={"predicted": "pred_b", "actual": "actual_b"})

        if bench.empty:
            continue

        for m in models:
            other = preds_long[(preds_long["model"] == m) & (preds_long["h"] == h)][
                ["date", "actual", "predicted"]
            ].rename(columns={"predicted": "pred_m"})

            merged = bench.merge(other, on=["date"], how="inner")

            # squared error loss (RMSE objective)
            actual = merged["actual_b"]
            # squared forecast error from candidate model
            loss_m = (actual - merged["pred_m"]) ** 2
            # squared forecast error from benchmark model
            loss_b = (actual - merged["pred_b"]) ** 2

            hac_lag = max(h - 1, 0)

            res = dm_test_hac(loss_m, loss_b, hac_lag=hac_lag)

            # Interpretation:
            # mean_d = mean(loss_m - loss_b)
            # if mean_d < 0 => model m has smaller loss => favours m
            favours = m if (res["mean_d"] < 0) else benchmark

            out_rows.append({
                "h": h,
                "model": m,
                "benchmark": benchmark,
                "hac_lag": hac_lag,
                "dm_stat": res["dm_stat"],
                "p_value": res["p_value"],
                "favours": favours
            })

    return pd.DataFrame(out_rows).sort_values(["h", "model"]).reset_index(drop=True)

In [21]:
lstm_df = load_lstm_preds(LSTM_PRED_PATH)
har_df = load_preds(HAR_PRED_PATH)
svr_df = load_preds(SVR_PRED_PATH)
xgb_df = load_preds(XGB_PRED_PATH)
lasso_df = load_preds(LASSO_PRED_PATH)
ridge_df = load_preds(RIDGE_PRED_PATH)
rf_df = load_preds(RF_PRED_PATH)

# print(lstm_df.head(3))
# print(har_df.head(3))
# print(svr_df.head(3))
# print(xgb_df.head(3))
# print(lasso_df.head(3))
# print(ridge_df.head(3))
# print(rf_df.head(3))

/var/folders/bg/l87j9pbx62sbccfkfg4fknp40000gn/T/ipykernel_29623/1920358959.py:7: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df["date"],dayfirst=True, errors = "raise")
/var/folders/bg/l87j9pbx62sbccfkfg4fknp40000gn/T/ipykernel_29623/1920358959.py:18: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df["date"],dayfirst=True, errors = "raise")
/var/folders/bg/l87j9pbx62sbccfkfg4fknp40000gn/T/ipykernel_29623/1920358959.py:18: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df["date"],dayfirst=True, errors = "raise")
/var/folders/bg/l87j9pbx62sbccfkfg4fknp40000gn/T/ipykernel_29623/1920358959.py:18: UserWa

In [ ]:
preds = pd.concat([lstm_df, har_df, svr_df, xgb_df,lasso_df,ridge_df, rf_df], ignore_index=True)

# Basic sanity checks
preds = preds.dropna(subset=["date", "h", "model", "actual", "predicted"])
preds = preds.sort_values(["h", "model", "date"]).reset_index(drop=True)

dm_results = dm_table_vs_benchmark(preds, benchmark=BENCHMARK, horizons=HORIZONS)
print("\nDiebold-Mariano test vs benchmark-", BENCHMARK)
print(dm_results)


Diebold-Mariano test vs benchmark- HAR-RV
    h         model benchmark  hac_lag   dm_stat   p_value       favours
0   1      HAR-RV-J    HAR-RV        0 -3.111166  0.001864      HAR-RV-J
1   1    HAR-RV-J-H    HAR-RV        0 -4.162568  0.000031    HAR-RV-J-H
2   1         LASSO    HAR-RV        0 -6.368450  0.000000         LASSO
3   1          LSTM    HAR-RV        0 -2.314413  0.020645          LSTM
4   1         RIDGE    HAR-RV        0 -4.874310  0.000001         RIDGE
5   1  RandomForest    HAR-RV        0 -7.468027  0.000000  RandomForest
6   1           SVR    HAR-RV        0 -5.041380  0.000000           SVR
7   1       XGBoost    HAR-RV        0 -7.073321  0.000000       XGBoost
8   3      HAR-RV-J    HAR-RV        2 -1.289544  0.197209      HAR-RV-J
9   3    HAR-RV-J-H    HAR-RV        2 -0.767728  0.442649    HAR-RV-J-H
10  3         LASSO    HAR-RV        2  0.511173  0.609230        HAR-RV
11  3          LSTM    HAR-RV        2  4.050986  0.000051        HAR-RV
12  3   

In [23]:
# Checker for stationarity
adf_results = adf_check_loss_diff(preds)  
print(adf_results)

    h         model benchmark    n    mean_d   adf_stat   adf_p_value  \
0   1      HAR-RV-J    HAR-RV  544 -0.036509 -13.745407  1.074902e-25   
1   1    HAR-RV-J-H    HAR-RV  544 -0.058063  -5.777762  5.211095e-07   
2   1         LASSO    HAR-RV  544 -0.189347  -7.265640  1.637812e-10   
3   1          LSTM    HAR-RV  544 -0.082299 -22.860148  0.000000e+00   
4   1         RIDGE    HAR-RV  544 -0.157857 -13.913294  5.460719e-26   
5   1  RandomForest    HAR-RV  544 -0.197188  -7.913086  3.900614e-12   
6   1           SVR    HAR-RV  544 -0.158509 -13.853599  6.932644e-26   
7   1       XGBoost    HAR-RV  544 -0.221465  -8.175993  8.374468e-13   
8   3      HAR-RV-J    HAR-RV  544 -0.003692 -21.402400  0.000000e+00   
9   3    HAR-RV-J-H    HAR-RV  544 -0.008262  -8.675115  4.437777e-14   
10  3         LASSO    HAR-RV  544  0.011376 -19.549648  0.000000e+00   
11  3          LSTM    HAR-RV  544  0.239790 -16.105875  5.073596e-29   
12  3         RIDGE    HAR-RV  544  0.080141  -6.39